# Cardiac Patient Monitoring System

## Data Preparation

**AI & ML Track — Final Project**

This notebook prepares the Cardiovascular Disease dataset for exploratory analysis and machine-learning experiments.

## Project Objective

Build a reproducible machine-learning workflow for a public cardiovascular dataset. This notebook focuses on loading, inspecting, cleaning, and defining the target variable before modeling.

## Dataset

The project uses the **Cardiovascular Disease dataset** selected from Kaggle. The classification target is `cardio`, where the task is to predict the target class from the available patient-related features.

> This project is for educational machine-learning analysis only. It does not provide clinical diagnosis, treatment recommendations, or emergency guidance.

## Step 1 — Environment and Imports

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

DATA_PATH = DATA_DIR / "cardio_train.csv"

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)


## Step 2 — Load the Dataset

The downloaded Kaggle CSV should be saved as `data/cardio_train.csv`. If your downloaded file has another name, rename it to `cardio_train.csv` before running this notebook.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download the Kaggle Cardiovascular Disease CSV and save it as data/cardio_train.csv."
    )

df = pd.read_csv(DATA_PATH, sep=";")
print(f"Dataset shape: {df.shape}")
df.head()


## Step 3 — Initial Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nFirst five rows:")
display(df.head())


## Step 4 — Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct.round(2)
})
display(missing_report)


## Step 5 — Duplicate Records

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

if duplicate_count:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Shape after removing duplicates:", df.shape)
else:
    print("No duplicate rows found.")


## Step 6 — Validate Numeric Ranges

We first inspect the minimum and maximum values instead of applying arbitrary clinical thresholds.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
range_report = df[numeric_cols].agg(["min", "max", "mean", "median"]).T
display(range_report)


## Step 7 — Handle Clearly Invalid Values

For this educational dataset, clearly invalid records are values that violate basic data constraints, such as non-positive height/weight or non-positive blood-pressure measurements. We also verify the target contains only the expected binary classes.

In [ ]:
before = len(df)

invalid_mask = (
    (df["height"] <= 0) |
    (df["weight"] <= 0) |
    (df["ap_hi"] <= 0) |
    (df["ap_lo"] <= 0) |
    (~df["cardio"].isin([0, 1]))
)

print("Clearly invalid rows:", int(invalid_mask.sum()))

df = df.loc[~invalid_mask].copy().reset_index(drop=True)
print("Rows before cleaning:", before)
print("Rows after cleaning:", len(df))


## Step 8 — Define the Target

`cardio` is the binary classification target. The identifier column `id` is not a predictive feature and will be removed before modeling.

In [ ]:
TARGET = "cardio"

print("Target:", TARGET)
print("Target classes:", sorted(df[TARGET].unique().tolist()))
print("\nClass counts:")
display(df[TARGET].value_counts().sort_index())

print("\nClass proportions:")
display(df[TARGET].value_counts(normalize=True).sort_index().rename("proportion"))


## Step 9 — Basic Feature Cleanup

Convert categorical variables to appropriate integer representations and remove the identifier.

In [ ]:
df = df.drop(columns=["id"], errors="ignore")

# Binary categorical variables are represented as integers.
for col in ["gender", "smoke", "alco", "active"]:
    if col in df.columns:
        df[col] = df[col].astype(int)

print("Final columns:")
print(df.columns.tolist())
print("\nFinal shape:", df.shape)
display(df.head())


## Step 10 — Save the Clean Dataset

In [ ]:
clean_path = DATA_DIR / "cardio_clean.csv"
df.to_csv(clean_path, index=False)

print(f"Clean dataset saved to: {clean_path}")


## Data Preparation Summary

- The dataset was loaded and inspected.
- Missing values and duplicates were checked.
- Clearly invalid values were removed using basic data constraints.
- The target variable `cardio` was verified as binary.
- The identifier `id` was removed because it is not a meaningful predictive feature.
- The cleaned dataset was saved for the following notebooks.